# Backtester Core Engine

**Owner:** James  
**Version:** 1.0 (Feb 6, 2026)  

Core backtesting engine for evaluating sports betting strategies.  
See `docs/reference/strategy-interface.md` for the strategy function contract.

## Output Format

| Column | Type | Description |
|--------|------|-------------|
| timestamp | datetime | When trade happened |
| game | str | "Home vs Away" |
| action | str | 'BUY_HOME' or 'BUY_AWAY' |
| bet_size | float | Dollars bet |
| odds | float | Decimal odds used |
| outcome | str | 'WIN' or 'LOSS' |
| pnl | float | Profit/loss for this trade |
| cumulative_pnl | float | Running total P&L |
| bankroll | float | Current bankroll after trade |

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any, Callable

import pandas as pd

## Configuration

In [ ]:
# Project root (one level up from tools/)
PROJECT_ROOT = Path("__file__").resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

# Default CSV paths
DUMMY_CSV = DATA_DIR / "dummy_backtest_input.csv"
TEST_CSV = DATA_DIR / "test_games.csv"

## `load_backtest_data()`

Loads game data for backtesting. Tries Railway database first, falls back to local CSV.

In [ ]:
def load_backtest_data(
    start_date: str,
    end_date: str,
    csv_path: str | Path | None = None,
) -> pd.DataFrame:
    """Load game data for backtesting.

    Attempts to load from Railway PostgreSQL database first. If DATABASE_URL
    is not set or the connection fails, falls back to loading from a local CSV.

    Args:
        start_date: Start date filter in "YYYY-MM-DD" format.
        end_date: End date filter in "YYYY-MM-DD" format.
        csv_path: Path to fallback CSV file. Defaults to dummy_backtest_input.csv.

    Returns:
        DataFrame with columns: timestamp, game, home_team, away_team,
        home_odds, away_odds, home_win. Sorted by timestamp ascending.

    Raises:
        FileNotFoundError: If no database and no CSV file found.

    Example:
        >>> data = load_backtest_data("2026-01-01", "2026-01-31")
        >>> print(data.columns.tolist())
        ['timestamp', 'game', 'home_team', 'away_team', 'home_odds', 'away_odds', 'home_win']
    """
    database_url = os.environ.get("DATABASE_URL")

    if database_url:
        try:
            from sqlalchemy import create_engine, text

            engine = create_engine(database_url)
            query = text("""
                SELECT
                    m.commence_time AS timestamp,
                    m.home_team || ' vs ' || m.away_team AS game,
                    m.home_team,
                    m.away_team,
                    o.home_odds,
                    o.away_odds,
                    m.home_win
                FROM sportsbook_matches m
                JOIN sportsbook_odds o ON m.id = o.match_id
                WHERE m.commence_time >= :start_date
                  AND m.commence_time <= :end_date
                ORDER BY m.commence_time ASC
            """)

            with engine.connect() as conn:
                df = pd.read_sql(query, conn, params={"start_date": start_date, "end_date": end_date})

            df["timestamp"] = pd.to_datetime(df["timestamp"])
            print(f"Loaded {len(df)} rows from Railway database.")
            return df

        except Exception as e:
            print(f"Database connection failed ({e}), falling back to CSV.")

    # Fallback to CSV
    if csv_path is None:
        csv_path = DUMMY_CSV
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"No CSV found at {csv_path}")

    df = pd.read_csv(csv_path, parse_dates=["timestamp"])

    # Filter by date range
    mask = (df["timestamp"] >= start_date) & (df["timestamp"] <= end_date)
    df = df.loc[mask].sort_values("timestamp").reset_index(drop=True)

    print(f"Loaded {len(df)} rows from {csv_path.name}.")
    return df

## `backtest()`

Core backtesting loop. Takes game data and a strategy function, simulates betting, and returns a trade log.

In [ ]:
def backtest(
    data: pd.DataFrame,
    strategy_fn: Callable[[pd.Series, dict[str, Any] | None], dict[str, Any]],
    initial_bankroll: float = 10000.0,
) -> pd.DataFrame:
    """Run a backtest over historical game data using a strategy function.

    Iterates through each game, calls the strategy function to get a signal,
    and simulates the bet outcome. Tracks bankroll, P&L, and trade history.

    Args:
        data: DataFrame from load_backtest_data() with columns: timestamp,
            game, home_team, away_team, home_odds, away_odds, home_win.
        strategy_fn: Callable matching the strategy interface. Takes
            (row: pd.Series, context: dict | None) and returns a dict with
            keys: action, confidence, size, reason (optional).
        initial_bankroll: Starting bankroll in dollars. Defaults to 10000.

    Returns:
        DataFrame with 9 columns: timestamp, game, action, bet_size, odds,
        outcome, pnl, cumulative_pnl, bankroll.

    Example:
        >>> data = load_backtest_data("2026-01-01", "2026-01-31")
        >>> results = backtest(data, always_bet_home)
        >>> print(results[["game", "action", "pnl", "bankroll"]])
    """
    bankroll = initial_bankroll
    cumulative_pnl = 0.0
    trades: list[dict[str, Any]] = []

    context: dict[str, Any] = {
        "initial_bankroll": initial_bankroll,
        "bankroll": bankroll,
        "trade_count": 0,
        "cumulative_pnl": 0.0,
    }

    for _, row in data.iterrows():
        if bankroll <= 0:
            break

        # Update context for strategy
        context["bankroll"] = bankroll
        context["trade_count"] = len(trades)
        context["cumulative_pnl"] = cumulative_pnl

        # Get strategy signal
        signal = strategy_fn(row, context)
        action = signal.get("action", "SKIP")

        if action == "SKIP":
            continue

        # Determine bet size (cap at current bankroll)
        bet_size = min(signal.get("size", 0.0), bankroll)
        if bet_size <= 0:
            continue

        # Determine odds based on action
        if action == "BUY_HOME":
            odds = row["home_odds"]
            won = row["home_win"] == 1
        elif action == "BUY_AWAY":
            odds = row["away_odds"]
            won = row["home_win"] == 0
        else:
            continue  # Invalid action, skip

        # Calculate P&L
        if won:
            pnl = bet_size * (odds - 1)
            outcome = "WIN"
        else:
            pnl = -bet_size
            outcome = "LOSS"

        cumulative_pnl += pnl
        bankroll += pnl

        trades.append({
            "timestamp": row["timestamp"],
            "game": row["game"],
            "action": action,
            "bet_size": round(bet_size, 2),
            "odds": odds,
            "outcome": outcome,
            "pnl": round(pnl, 2),
            "cumulative_pnl": round(cumulative_pnl, 2),
            "bankroll": round(bankroll, 2),
        })

    return pd.DataFrame(trades)

## Example Strategy: Always Bet Home

A simple strategy that always bets on the home team with a fixed stake. Used for testing the backtester.

In [ ]:
def always_bet_home(
    row: pd.Series,
    context: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Example strategy: always bet on the home team.

    Bets a fixed $100 on the home team for every game.
    This is a naive strategy used to validate the backtester.

    Args:
        row: Game data row with home_odds, away_odds, etc.
        context: Optional backtester context (bankroll, trade count, etc.).

    Returns:
        Signal dict with action='BUY_HOME', confidence=0.5, size=100.
    """
    return {
        "action": "BUY_HOME",
        "confidence": 0.5,
        "size": 100.0,
        "reason": "Always bet home (test strategy)",
    }

## End-to-End Test Run

Load dummy data and run the backtester with the "always bet home" strategy.

In [ ]:
# Load dummy data
data = load_backtest_data("2026-01-01", "2026-01-31")
print(f"Games loaded: {len(data)}")
data.head()

In [ ]:
# Run backtest
results = backtest(data, always_bet_home, initial_bankroll=10000.0)
print(f"Total trades: {len(results)}")
print(f"Columns: {results.columns.tolist()}")
results

In [ ]:
# Summary statistics
if len(results) > 0:
    wins = (results["outcome"] == "WIN").sum()
    losses = (results["outcome"] == "LOSS").sum()
    win_rate = wins / len(results)
    final_pnl = results["cumulative_pnl"].iloc[-1]
    final_bankroll = results["bankroll"].iloc[-1]

    print(f"Win Rate:        {win_rate:.1%} ({wins}W / {losses}L)")
    print(f"Total P&L:       ${final_pnl:,.2f}")
    print(f"Final Bankroll:  ${final_bankroll:,.2f}")
    print(f"ROI:             {final_pnl / 10000:.1%}")
else:
    print("No trades executed.")

## Verify Output Format

Confirm the results DataFrame has exactly the 9 required columns with correct types.

In [ ]:
# Validate output schema
REQUIRED_COLUMNS = [
    "timestamp", "game", "action", "bet_size",
    "odds", "outcome", "pnl", "cumulative_pnl", "bankroll",
]

assert results.columns.tolist() == REQUIRED_COLUMNS, (
    f"Column mismatch!\n"
    f"Expected: {REQUIRED_COLUMNS}\n"
    f"Got:      {results.columns.tolist()}"
)

assert len(results) > 0, "No trades were produced"
assert all(results["action"].isin(["BUY_HOME", "BUY_AWAY"])), "Invalid action values"
assert all(results["outcome"].isin(["WIN", "LOSS"])), "Invalid outcome values"
assert all(results["bet_size"] > 0), "Bet sizes must be positive"

print("All output format checks passed!")